# v2 Export Submission Notebook

This notebook converts model prediction files (JSONL) into the competition
submission format required by ArgMining 2026.

### Outputs
| File | Description |
|---|---|
| `submission_per_doc/<TEXT_ID>.json` | One JSON file per document |
| `submission_all.json` | All docs as a list |
| `submission_all_dict.json` | All docs keyed by TEXT_ID (bonus) |
| `submission.zip` | Final ZIP for upload |

### Submission JSON schema (per doc)
```
{
  "TEXT_ID": "...",
  "METADATA": {
    "structure": {
      "preambular_paras": [int, ...],
      "operative_paras":  [int, ...],
      "think":            "..."
    }
  },
  "body": {
    "paras": [
      {
        "para_number": int,
        "para":        "...",
        "type":        "preambular" | "operative",
        "tags":        ["..."],
        "matched_paras": {"<pid>": "<relation>"},
        "think":       "..."
      },
      ...
    ]
  }
}
```

### matched_paras flattening rule
Internal representation may store a list of relations per link.
For submission, exactly **one** relation string is required per link.
Priority: `modifying > contradictive > supporting > complemental`.


In [ ]:
# Install / verify dependencies (run once)
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for pkg in ["pandas"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        _pip(pkg)

print("Dependencies OK.")


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
from pathlib import Path

# Root data directory
BASE_DIR = Path("/root/autodl-fs")

# Input: clean pseudo-labels produced by v2_pseudo_label_coreset.ipynb
# Can also point at any other JSONL with the same schema.
PRED_JSONL   = BASE_DIR / "pseudo_labels_v2_clean.jsonl"

# Optionally also merge the _fixed file
FIXED_JSONL  = BASE_DIR / "pseudo_labels_v2_fixed.jsonl"

# Output paths
OUT_PER_DOC_DIR = BASE_DIR / "submission_per_doc"
OUT_ALL_LIST    = BASE_DIR / "submission_all.json"
OUT_ALL_DICT    = BASE_DIR / "submission_all_dict.json"
OUT_ZIP         = BASE_DIR / "submission.zip"

# Relation priority for flattening (modifying wins over complemental etc.)
REL_PRI = ["modifying", "contradictive", "supporting", "complemental"]

print("Configuration loaded.")
print("  PRED_JSONL  :", PRED_JSONL)
print("  OUT_ZIP     :", OUT_ZIP)


In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────
import json

def flatten_rel_list(rels) -> str | None:
    """
    Given a list (or single string) of relations, return the highest-priority one.
    Priority: modifying > contradictive > supporting > complemental.
    """
    if isinstance(rels, str):
        return rels if rels in REL_PRI else None
    if not rels:
        return None
    for r in REL_PRI:
        if r in rels:
            return r
    return rels[0] if rels else None


def flatten_matched_paras(mp_internal: dict) -> dict:
    """
    Convert internal matched_paras {pid: [rel, ...] | rel}
    → submission format {pid: "rel"} (single relation string).
    """
    if not isinstance(mp_internal, dict):
        return {}
    out = {}
    for pid_k, v in mp_internal.items():
        rel = flatten_rel_list(v)
        if rel:
            out[str(int(pid_k))] = rel
    return out


def doc_level_think(preambular_paras: list, operative_paras: list) -> str:
    """Generate a deterministic doc-level think string."""
    return (
        f"Type assignment used French trigger word priors; "
        f"preambular_paras={len(preambular_paras)}, "
        f"operative_paras={len(operative_paras)}. "
        f"Relations: modifying chosen when scope/conditions explicitly changed, "
        f"complemental for parallel-info additions."
    )


print("Helper functions defined.")


In [ ]:
# ── Load prediction records from JSONL ────────────────────────────────────────
import json

def load_jsonl(path: Path) -> list:
    """Load a JSONL file into a list of dicts. Returns [] if file not found."""
    if not path.exists():
        print(f"  (not found, skipping) {path}")
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  WARNING: JSON error on line {i}: {e}")
    print(f"  Loaded {len(records)} records from {path}")
    return records

all_records_raw = []
print("Loading main predictions …")
all_records_raw.extend(load_jsonl(PRED_JSONL))
print("Loading fixed predictions …")
all_records_raw.extend(load_jsonl(FIXED_JSONL))

# Deduplicate: later record (fixed) overrides earlier (main)
deduped = {}
for rec in all_records_raw:
    key = (str(rec["doc_id"]), str(rec["pid"]))
    deduped[key] = rec

all_records = list(deduped.values())
print(f"\nTotal unique (doc_id, pid) pairs: {len(all_records)}")


In [ ]:
# ── Group records by doc_id ────────────────────────────────────────────────────
from collections import defaultdict

by_doc = defaultdict(list)
for rec in all_records:
    by_doc[str(rec["doc_id"])].append(rec)

# Sort each doc's paragraphs by pid
for doc_id in by_doc:
    by_doc[doc_id].sort(key=lambda r: float(r["pid"]))

print(f"Documents found: {len(by_doc)}")


In [ ]:
# ── Build the submission JSON object for one document ─────────────────────────

def build_doc_json(doc_id: str, items: list) -> dict:
    """
    Convert a list of annotation records for one document into the
    competition submission JSON schema.

    Parameters
    ----------
    doc_id : str
        Document identifier.
    items  : list
        Records sorted by pid (ascending).  Each record must have:
          doc_id, pid, text_fr, annotation (type, tags, matched_paras, think)
    """
    preambular_paras, operative_paras = [], []
    paras_out = []

    for rec in items:
        ann        = rec.get("annotation", {})
        pid        = int(rec["pid"])
        text_fr    = str(rec.get("text_fr", ""))
        para_type  = str(ann.get("type", "preambular"))
        tags       = ann.get("tags", [])
        if not isinstance(tags, list):
            tags = [tags] if tags else []
        matched    = flatten_matched_paras(ann.get("matched_paras", {}))
        think      = str(ann.get("think", ""))

        if para_type == "operative":
            operative_paras.append(pid)
        else:
            preambular_paras.append(pid)

        paras_out.append({
            "para_number":    pid,
            "para":           text_fr,
            "type":           para_type,
            "tags":           tags,
            "matched_paras":  matched,
            "think":          think,
        })

    return {
        "TEXT_ID": doc_id,
        "METADATA": {
            "structure": {
                "preambular_paras": preambular_paras,
                "operative_paras":  operative_paras,
                "think":            doc_level_think(preambular_paras, operative_paras),
            }
        },
        "body": {
            "paras": paras_out,
        },
    }


print("build_doc_json defined.")


In [ ]:
# ── Export per-doc JSON files ─────────────────────────────────────────────────
import json
from tqdm.auto import tqdm

OUT_PER_DOC_DIR.mkdir(parents=True, exist_ok=True)

doc_jsons = {}   # doc_id → doc JSON object (reused for merged outputs)

for doc_id, items in tqdm(by_doc.items(), desc="exporting per-doc"):
    doc_obj = build_doc_json(doc_id, items)
    doc_jsons[doc_id] = doc_obj

    out_path = OUT_PER_DOC_DIR / f"{doc_id}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(doc_obj, f, ensure_ascii=False, indent=2)

print(f"\nPer-doc JSON files written: {len(doc_jsons)}")
print(f"Directory: {OUT_PER_DOC_DIR}")


In [ ]:
# ── Export merged submission files ────────────────────────────────────────────

# 1. List of docs (primary format)
all_list = sorted(doc_jsons.values(), key=lambda d: str(d["TEXT_ID"]))
with open(OUT_ALL_LIST, "w", encoding="utf-8") as f:
    json.dump(all_list, f, ensure_ascii=False, indent=2)
print(f"Merged list written : {OUT_ALL_LIST}  ({len(all_list)} docs)")

# 2. Dict keyed by TEXT_ID (bonus / alternative lookup format)
all_dict = {str(d["TEXT_ID"]): d for d in all_list}
with open(OUT_ALL_DICT, "w", encoding="utf-8") as f:
    json.dump(all_dict, f, ensure_ascii=False, indent=2)
print(f"Merged dict written : {OUT_ALL_DICT}")


In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────────
import random

print("=== Submission sanity checks ===\n")

# 1. Schema validation
required_doc_keys  = {"TEXT_ID", "METADATA", "body"}
required_para_keys = {"para_number", "para", "type", "tags", "matched_paras", "think"}
schema_errors = []

for doc_id, doc_obj in doc_jsons.items():
    missing_doc = required_doc_keys - set(doc_obj.keys())
    if missing_doc:
        schema_errors.append(f"{doc_id}: missing doc keys {missing_doc}")
    for p in doc_obj.get("body", {}).get("paras", []):
        missing_p = required_para_keys - set(p.keys())
        if missing_p:
            schema_errors.append(f"{doc_id} para {p.get('para_number')}: missing {missing_p}")
        # matched_paras values must be strings
        for pid_k, rel in p.get("matched_paras", {}).items():
            if not isinstance(rel, str):
                schema_errors.append(
                    f"{doc_id} para {p.get('para_number')}: "
                    f"matched_paras[{pid_k}] is {type(rel).__name__}, expected str")

if schema_errors:
    print("SCHEMA ERRORS:")
    for e in schema_errors[:20]:
        print(" ", e)
else:
    print("✓ All documents pass schema validation.")

# 2. Type distribution
from collections import Counter
type_dist = Counter()
rel_dist  = Counter()
for doc_obj in doc_jsons.values():
    for p in doc_obj["body"]["paras"]:
        type_dist[p["type"]] += 1
        for rel in p["matched_paras"].values():
            rel_dist[rel] += 1

print(f"\nType distribution : {dict(type_dist)}")
print(f"Relation distribution: {dict(rel_dist)}")

# 3. Sample one doc
sample_doc_id = random.choice(list(doc_jsons.keys()))
sample_doc    = doc_jsons[sample_doc_id]
print(f"\nSample doc: {sample_doc_id}")
print(f"  preambular_paras : {sample_doc['METADATA']['structure']['preambular_paras'][:5]} ...")
print(f"  operative_paras  : {sample_doc['METADATA']['structure']['operative_paras'][:5]} ...")
print(f"  nb_paras         : {len(sample_doc['body']['paras'])}")
if sample_doc["body"]["paras"]:
    p0 = sample_doc["body"]["paras"][0]
    print(f"  first para       : {p0}")


In [ ]:
# ── Package everything into submission.zip ────────────────────────────────────
import zipfile, os

with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Per-doc files → submission_per_doc/<TEXT_ID>.json
    for json_file in sorted(OUT_PER_DOC_DIR.glob("*.json")):
        arcname = f"submission_per_doc/{json_file.name}"
        zf.write(json_file, arcname)

    # Merged files
    if OUT_ALL_LIST.exists():
        zf.write(OUT_ALL_LIST, "submission_all.json")
    if OUT_ALL_DICT.exists():
        zf.write(OUT_ALL_DICT, "submission_all_dict.json")

# Report contents
with zipfile.ZipFile(OUT_ZIP, "r") as zf:
    names = zf.namelist()
    total_size = sum(info.file_size for info in zf.infolist())

print(f"submission.zip created: {OUT_ZIP}")
print(f"  {len(names)} files  |  uncompressed {total_size/1024:.1f} KB")
print("  First 10 entries:")
for n in names[:10]:
    print(f"    {n}")
if len(names) > 10:
    print(f"    ... and {len(names)-10} more")


In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print("=== Submission Export Summary ===")
print(f"  Documents exported : {len(doc_jsons)}")
print(f"  Per-doc dir        : {OUT_PER_DOC_DIR}")
print(f"  Merged list JSON   : {OUT_ALL_LIST}")
print(f"  Merged dict JSON   : {OUT_ALL_DICT}")
print(f"  ZIP package        : {OUT_ZIP}")
print()
print("Upload submission.zip to the competition platform.")
